In [ ]:
import os
from os.path import join, isfile

import json
import re
import pandas as pd
import plotly.express as px
import networkx as nx

import pickle as pkl
import scipy.sparse as sp

In [ ]:
# Path to the folder containing your JSON files
folder_path = '/mnt/43b8df78-b48a-4d67-b3aa-8c9e00a3ecbd/Research/Results'
DS_PATH = "/mnt/43b8df78-b48a-4d67-b3aa-8c9e00a3ecbd/Research/CurrentExperiment"

In [ ]:
data_list = []

# Regex to capture budget and infected nodes from the filename
# Format: result_MeshShield_BUDGET_graph_10300400_INFECTED.json
file_pattern = re.compile(r"result_MeshShield_(\d+)_graph_10300400_(\d+)\.json")

for filename in os.listdir(folder_path):
    match = file_pattern.match(filename)
    if match:
        budget = int(match.group(1))
        infected_nodes = int(match.group(2))
        
        file_path = os.path.join(folder_path, filename)
        
        try:
            with open(file_path, 'r') as f:
                content = json.load(f)
                runtime = content.get("Total time")
                
                data_list.append({
                    "Budget": budget,
                    "Infected_Nodes": infected_nodes,
                    "Runtime": runtime
                })
        except (json.JSONDecodeError, IOError) as e:
            print(f"Error reading {filename}: {e}")

# Create a DataFrame for easy analysis
df = pd.DataFrame(data_list)

# Sort the data for logical plotting
df = df.sort_values(by=["Budget", "Infected_Nodes"])

print(df.head())

In [ ]:
# 1. Ensure columns are numeric
df['Budget'] = pd.to_numeric(df['Budget'])
df['Infected_Nodes'] = pd.to_numeric(df['Infected_Nodes'])
df['Runtime'] = pd.to_numeric(df['Runtime'])

# 2. Sort by Legend (Infected_Nodes) then X-axis (Budget)
df = df.sort_values(by=['Infected_Nodes', 'Budget'])

# 3. Create a string column for the legend
df['Infected_Nodes_Str'] = df['Infected_Nodes'].astype(str)

# -------------------------------------------------------
# Enhanced Plot with BETTER COLORS
# -------------------------------------------------------
fig = px.line(
    df, 
    x="Budget", 
    y="Runtime", 
    color="Infected_Nodes_Str",
    markers=True,
    # SWITCHING TO A BETTER COLOR PALETTE HERE:
    color_discrete_sequence=px.colors.qualitative.Bold, 
    title="Algorithm Runtime vs Immunization Budget",
    labels={
        "Infected_Nodes_Str": "Infected Nodes", 
        "Runtime": "Total Time (s)", 
        "Budget": "Immunization Budget"
    }
)

# --- STYLING UPDATES FOR LEGIBILITY ---
fig.update_layout(
    # INCREASED: Global Font Settings (Size 22)
    font=dict(size=22, family="Arial"),
    
    title=dict(
        # INCREASED: Title font size
        font=dict(size=30, weight="bold"), 
        y=0.96
    ),
    
    # --- NEW LEGEND SETTINGS ---
    legend=dict(
        title=dict(text="Infected Nodes", font=dict(size=24, weight="bold")),
        font=dict(size=22),
        
        # Horizontal & Below Chart
        orientation="h",
        yanchor="top", 
        y=-0.2, # Pushed down slightly more to clear x-axis
        xanchor="center", 
        x=0.5,
        
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="Black",
        borderwidth=1
    ),
    
    # Add bottom margin for the legend
    margin=dict(b=140),
    
    width=1200, # Increased width
    height=900, # Increased height
    
    # Make the background white for maximum contrast
    plot_bgcolor='white',
    paper_bgcolor='white'
)

# Add grid lines back in (since we set background to white)
fig.update_xaxes(
    title_font=dict(size=26, weight="bold"), # Increased
    tickfont=dict(size=22),                  # Increased
    showgrid=True, gridwidth=1, gridcolor='LightGray',
    zeroline=True, zerolinewidth=2, zerolinecolor='Black' # Stronger axis line
)
fig.update_yaxes(
    title_font=dict(size=26, weight="bold"), # Increased
    tickfont=dict(size=22),                  # Increased
    showgrid=True, gridwidth=1, gridcolor='LightGray',
    zeroline=True, zerolinewidth=2, zerolinecolor='Black'
)

fig.update_traces(
    line=dict(width=5), # Thicker lines
    marker=dict(size=14, line=dict(width=2, color='DarkSlateGrey')) # Larger markers
)

fig.show()
fig.write_image("justTime.png", 
                    width=1200, height=900, scale=2,  # High Res
                    format="png")

In [ ]:
all_graphs = [f for f in os.listdir(DS_PATH) if (isfile(join(DS_PATH, f)) and (f.endswith(".pkl") or f.endswith(".npz")))]

In [ ]:
for graph in all_graphs:
    graph_path = join(DS_PATH, graph)

    G = pkl.load(open(graph_path, 'rb')) if graph.endswith('pkl') else sp.load_npz(graph_path)
    G = nx.from_scipy_sparse_array(G) if sp.issparse(G) else G.copy()
    num_nodes = G.number_of_nodes()
    num_edges = G.number_of_edges()

    print(f"{graph}: {num_nodes} nodes and {num_edges} edges.")
